# 1. Business hypothesis and experiment overview

**Experiment:** `3.1_pretrained_similarity` (provisional unused pretrained-track ID; confirm with the group). **Owner:** set below.

**Hypothesis:** Frozen shared multilingual text embeddings can connect a candidate article to a reader's historical interests, including eligible articles that were never in the recorded slate. This should improve recovery of known clicked articles in a catalogue-derived top-100 shortlist, supporting personalised exploration before business reranking.

Train an incremental logistic classifier on the **existing shared shown/clicked pairs**. Select its regularisation/checkpoint by **catalogue known-click Recall@100**, breaking ties with **observed-click nDCG@100**, then earlier evaluation. This is a new model and evaluation setup, not a renamed version of the old local sampled logistic regression.

Use a declared catalogue-safe subset of the shared raw features plus normalised mean-history text similarity. The text encoder is frozen; the train-only imputer, scaler and logistic classifier are trained. Read the shared embedding provenance rather than assuming these are the earlier E5 vectors.

The default is a bounded **development** run, with complete recorded slates and a full eligible catalogue for every sampled request. Audit mode is the default. It does not claim a full export or any completed real-data evaluation. The supplied official test is unlabelled; export it without evaluating or tuning on it.


# 2. Setup and run configuration

In Colab, mount Drive and make this experiment's `catalogue_pipeline.py` available. Opening a notebook from GitHub does not automatically download sibling Python files. The next cell finds a local checkout or prompts you to upload that helper file. No embedding API calls are made.

Run setup and the audit first. Set `RUN_PHASE = "run"` only after inspecting the data and costs. Use a new run ID for changed inputs, seeds or configurations. For a new runtime with a saved model, set `TRAIN_MODEL = False` and point `LOAD_RUN` to the completed persistent run.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import hashlib
import importlib.util
import json
import subprocess
import sys

try:
    IN_COLAB = importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

EXPERIMENT_ID = '3.1_pretrained_similarity'
OWNER = ''  # Fill in your name before final handoff.
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder')
# For local downloaded data, change DRIVE_ROOT above.

candidates = [Path.cwd(), Path.cwd() / 'Experiments' / EXPERIMENT_ID]
CODE_DIR = next((p for p in candidates if (p / 'catalogue_pipeline.py').is_file()), None)
if CODE_DIR is None and IN_COLAB:
    from google.colab import files
    print('Upload catalogue_pipeline.py from Experiments/3.1_pretrained_similarity.')
    uploaded = files.upload()
    if 'catalogue_pipeline.py' not in uploaded:
        raise FileNotFoundError('The experiment helper catalogue_pipeline.py is required.')
    CODE_DIR = Path('/content/Experiments') / EXPERIMENT_ID
    CODE_DIR.mkdir(parents=True, exist_ok=True)
    (CODE_DIR / 'catalogue_pipeline.py').write_bytes(uploaded['catalogue_pipeline.py'])
if CODE_DIR is None:
    raise FileNotFoundError('Run from the repository root or the experiment folder, or set CODE_DIR.')

# Install the same declared dependencies in this notebook's Python environment.
subprocess.run([sys.executable, '-m', 'pip', 'install',
                'numpy>=1.26', 'pandas>=2.2', 'pyarrow>=15',
                'scikit-learn>=1.4', 'joblib>=1.3', 'matplotlib>=3.8'], check=True)
sys.path.insert(0, str(CODE_DIR))
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import catalogue_pipeline as cp


In [ ]:
SEED = 42
FEATURE_CASE = 'raw'
TRAIN_MODEL = True
RUN_PHASE = 'audit'  # 'audit' first, then 'run'; no training/export in audit mode.
# Keep this ID stable when switching this runtime from audit to run.
# Assign RUN_ID = None explicitly when starting a fresh experiment.
RUN_ID = globals().get('RUN_ID') or (datetime.now(ZoneInfo('Asia/Singapore')).strftime('%Y%m%d_%H%M%S') + f'_seed{SEED}')
WORK = CODE_DIR / 'outputs' / RUN_ID
PERSISTENT_ROOT = DRIVE_ROOT / 'Experiments'
LOAD_RUN = None  # E.g. PERSISTENT_ROOT / EXPERIMENT_ID / 'a_completed_run_id'

CONFIG = {
    'experiment_id': EXPERIMENT_ID, 'owner': OWNER, 'run_id': RUN_ID,
    'drive_root': str(DRIVE_ROOT), 'seed': SEED, 'feature_case': FEATURE_CASE,
    'scope': 'development',
    'input_version': None,  # Shared preprocessing release/version, when known.
    'embedding_source': {'artifact': 'shared text_embeddings.parquet',
                         'model': None, 'revision': None},  # Confirm with the preprocessing owner.
    'embedding_column': 'text_embedding', 'embedding_dimension': None,  # Inferred and validated.
    'train_requests': 5000, 'selection_requests': 100,
    'export_requests': {'train': 25, 'validation': 25, 'test': 25},
    'alphas': [0.0001, 0.001], 'epochs': 3, 'patience': 2,
    'minimum_recall_improvement': 0.0,
    'batch_size': 8192, 'catalogue_batch_size': 4096,
    'hash_inputs': False,  # True computes full file SHA256; False records size/mtime/schema.
    'max_index_gb': 20, 'max_export_pairs': 10_000_000,
    'max_export_gb': 10, 'max_scoring_seconds': 3600,
    'persistent_root': str(PERSISTENT_ROOT),
    'paths': {},  # Override ambiguous/moved paths relative to DRIVE_ROOT or with absolute paths.
}
# A final run must use scope='final', train_requests=None, and export_requests
# {'train': None, 'validation': None, 'test': None}. Confirm owner, input_version,
# actual embedding encoder identity and group catalogue policy, then inspect
# full costs before raising any budget. Do not rename sampled outputs as final.

CONFIG['code_provenance'] = {
    'helper_sha256': hashlib.sha256((CODE_DIR / 'catalogue_pipeline.py').read_bytes()).hexdigest(),
    'commit': None, 'uncommitted_changes': None,
}
try:
    commit = subprocess.run(['git', '-C', str(CODE_DIR), 'rev-parse', 'HEAD'],
                            capture_output=True, text=True, check=True).stdout.strip()
    dirty = subprocess.run(['git', '-C', str(CODE_DIR), 'status', '--porcelain'],
                           capture_output=True, text=True, check=True).stdout
    CONFIG['code_provenance'].update(commit=commit, uncommitted_changes=bool(dirty))
except (subprocess.CalledProcessError, FileNotFoundError):
    pass  # Uploaded helpers have a content hash even without a Git checkout.
print('Working run:', WORK)
print('Persistent destination:', PERSISTENT_ROOT / EXPERIMENT_ID / RUN_ID)


# 3. Load and check shared data

Resolve actual shared paths and inspect Parquet schemas, source versions, original composite keys, true labels, decoded vector dimensions and flags. The helper rejects ambiguous paths, duplicate candidates, malformed vectors, non-null official test targets, inconsistent request features and any mismatch between shared article features and catalogue reconstruction.

This step disk-indexes recorded candidates and requests for complete-slate sampling and diagnostics. It does not create catalogue-wide negatives. Profile lookup is disk-backed with a bounded memory cache. Source/input checks can take time on large shared files.

**Declared eligibility policy:** ordered union of train/validation and test catalogue snapshots, first snapshot per ID retained; article publication time must be known and no later than the request. No article-age cutoff or premium/access exclusion is applied. Confirm the same version/policy with the group before comparing models. Missing vectors do not remove articles. Missing publication times are excluded because availability cannot be established, and excluded known clicks stay in recall denominators.


In [ ]:
PREPARED = cp.prepare(CONFIG, WORK)
audit = PREPARED['audit']
print('Catalogue articles:', audit['catalogue_articles'])
print('Articles without usable vectors:', audit['catalogue_articles_without_vectors'])
print('Decoded embedding dimension:', audit['embedding_info']['dimension'])
print('Catalogue eligibility policy:', audit['eligibility_policy'])
pd.DataFrame(audit['request_counts']).T


In [ ]:
planned = pd.DataFrame(audit['planned_exports']).T
full = pd.DataFrame(audit['full_export_costs']).T
print('PLANNED export costs (sampled development by default):')
display(planned[['requests', 'eligible_pairs', 'estimated_csv_bytes', 'empty_pools']])
print('FULL split export costs; these are not the planned development outputs:')
display(full[['requests', 'eligible_pairs', 'estimated_csv_bytes', 'empty_pools']])
print('Disk-index planning estimate, GB:', round(audit['estimated_index_bytes'] / 1e9, 2))
print('CSV estimates assume 130 bytes per pair; actual sizes vary.')
print('Runtime is estimated from measured selection scoring before exports start.')
cp.check_cost(PREPARED)  # Refuses a run beyond declared pair/disk limits.


# 4. Prepare model inputs

The model reads the existing shared candidate rows directly. It uses the following **declared subset**, not all 202 historical predictors:

- Request context: hour, weekday, weekend, subscription/SSO/front-page flags and history size, taken unchanged from each shared request.
- Article metadata: premium, body word count, deduplicated topic count, topic-presence flag, article age and age-missing flag. The catalogue builder reproduces and audits these against the shared raw table.
- Frozen text similarity: cosine of candidate text embedding with the normalised average of eligible past-click embeddings, plus candidate/history missingness and history-vector coverage.

All other shared predictors are omitted and the model is retrained. In particular, original-slate size and slate-relative age are not silently reused or filled for unshown articles. Numerical missing values are imputed with train-only means, with a missingness indicator per feature, then scaled using train-only statistics. Features entirely missing during training have zero fallback plus an explicit flag.

Historical click IDs/times are additional declared inputs. History is fixed strictly before the earliest request in each split; a temporal violation raises an error. Repeated clicks retain weight. A missing history row/vector has explicit fallback flags. No current or subsequent reading outcomes, candidate-list position, user/article IDs or `clicked` values are predictors.


In [ ]:
print('Ordered model inputs:', cp.FEATURES)
print('Shared source columns:', cp.REQUEST + cp.ARTICLE)
print('Derived frozen-embedding features:', cp.SEMANTIC)
print('Validation selection requests:', len(PREPARED['selection_requests']))


# 5. Define and train / load model

Use `SGDClassifier(loss='log_loss')` for logistic training in batches, so all shared training pairs can be processed without materialising one giant matrix. The encoder remains frozen. This is intentionally different from the earlier in-memory logistic estimator.

The predeclared search has two regularisation strengths and at most three passes each. Every epoch evaluates the same fixed validation-request sample against its **full eligible catalogue**. Select by known-click Recall@100, then observed-click nDCG@100, then earlier checkpoint. Patience counts these selection evaluations. AUC and training loss never choose the checkpoint.

The selected inference model is saved locally and checkpointed to a separate Drive `_in_progress` folder while training. That folder is not a complete handoff bundle. Exact training resumption is not implemented. The final complete bundle is copied and reloaded in sections 7-8.


In [ ]:
if PREPARED['work'] != WORK or PREPARED['config'] != CONFIG:
    raise RuntimeError('Rerun sections 3-4 after changing the run configuration.')

if RUN_PHASE == 'run':
    if TRAIN_MODEL:
        training = cp.train(PREPARED)
    else:
        if LOAD_RUN is None:
            raise ValueError('Set LOAD_RUN to the completed persistent source run.')
        cp.load_trained_run(PREPARED, LOAD_RUN)
        training = json.loads((WORK / 'training_summary.json').read_text())
    print('Selected checkpoint:', training['selected'])
else:
    print('Audit only. Inspect the inputs/costs, then set RUN_PHASE="run" and run this cell.')


# 6. Validation diagnostics and explainability

Catalogue Recall@100 is the mean fraction of each request's **original** known clicks retained in its top 100. Clicks excluded by eligibility remain in that denominator. Observed-click nDCG@100 likewise uses the original click set in its ideal ranking. Empty pools with known clicks receive zero. Requests without known clicks are counted separately. Unshown outcomes remain unknown; zero observed retrieval gain is not a negative training label.

Tie order is descending score then ascending original article ID, including ties crossing rank 100. Keep exposure bias explicit. These metrics measure recovery of recorded clicks, not unbiased catalogue relevance or retention. Validation is development data once repeatedly used for selection. The final official test has no metrics.

Inspect the logged learning/selection curves and coefficients as a focused check of the hypothesis. A coefficient or linear contribution is an association, not a causal explanation. An exact explained validation top-five is saved during export.


In [ ]:
if RUN_PHASE == 'run':
    trials_path = WORK / 'tuning/trials.csv'
    if trials_path.exists():
        trials = pd.read_csv(trials_path)
        display(trials)
        fig, axes = plt.subplots(1, 2, figsize=(11, 4))
        for trial_id, rows in trials[trials.status.eq('complete')].groupby('trial'):
            axes[0].plot(rows.epoch, rows.training_loss_after_batch_updates, marker='o', label=f'Trial {trial_id}')
            axes[1].plot(rows.epoch, rows['validation.known_click_recall@100'], marker='o', label=f'Trial {trial_id}')
        axes[0].set(title='In-sample optimisation diagnostic', xlabel='Epoch', ylabel='After-update batch log loss')
        axes[1].set(title='Catalogue selection objective', xlabel='Epoch', ylabel='Known-click Recall@100')
        for ax in axes: ax.legend()
        plt.tight_layout()
    display(pd.json_normalize(training['selected']))


In [ ]:
if RUN_PHASE == 'run':
    selected = joblib.load(WORK / 'model/selected.joblib')
    names = cp.FEATURES + [name + '__missing' for name in cp.FEATURES]
    coefficients = pd.DataFrame({'feature': names, 'coefficient': selected['classifier'].coef_[0]})
    display(coefficients.assign(magnitude=lambda f: f.coefficient.abs()).sort_values('magnitude', ascending=False).head(12))


# 7. Predict and export all splits

The selected model scores the eligible catalogue for every **declared export request** in train, validation and test. Full scores are streamed, not restricted to the top 100. Export composite original-ID `behaviour_id = user_id:impression_id`, score logits and genuine uncalibrated logistic probabilities. Probabilities need not sum to one.

For each split, save `predictions/{split}/submission_click_probabilities.csv` and `rankings/{split}/submission_top_100.csv`, plus separate `diagnostics/recorded_slates/{split}_scores.csv`. Preserve empty pools as `[]`. Validate exact catalogue membership, future exclusions, every original diagnostic candidate, unique ordered top-100 IDs and finite scores/probabilities.

**Development mode exports are sampled and incomplete for final handoff.** To cover all agreed requests, use final scope, all training requests and `None` export limits, confirm input/encoder provenance, inspect the full pair/disk estimates and measured runtime before increasing budgets. Batching does not remove the cost of billions of exported rows. If full export is infeasible, document it instead of silently restricting the catalogue.


In [ ]:
RESULT = None
if RUN_PHASE == 'run':
    # The selected model already exists; this call does not fit again.
    RESULT = cp.execute(PREPARED, train_model=False)
    display(pd.json_normalize(RESULT['metrics']))
    print('Metrics for the declared exported validation requests:')
    display(pd.json_normalize(RESULT['exported_validation']))
    print('Export checks:', RESULT['checks'])
    print('Complete final handoff:', RESULT['handoff_complete'])
    print('Persistent copy:', RESULT['persistence'])


# 8. Reload check

The exporter copies the complete run folder to the declared persistent Drive destination, checks file coverage/sizes and reloads the selected model/preprocessing from that copy. It predicts a saved raw-feature probe and checks logits/probabilities within `rtol=1e-7, atol=1e-8` without fitting. Raw catalogue feature construction was audited against the shared source rows before training; the probe specifically verifies the saved estimator and preprocessing.

A teammate can use `TRAIN_MODEL=False` with the completed persistent `LOAD_RUN`. Source versions/paths and feature/pool contracts must still match. An incomplete `_in_progress` checkpoint is not that load-only bundle. Schema/score checks on test are label-free; do not invent test metrics.


In [ ]:
if RESULT is not None:
    persistent = RESULT['persistence']
    if persistent['persistent_copy_complete']:
        saved_run = Path(persistent['persistent_location'])
        saved_model = joblib.load(saved_run / 'model/selected.joblib')
        with np.load(saved_run / 'model/reload_probe.npz', allow_pickle=False) as probe:
            score, probability = cp.predict(saved_model, probe['raw'])
            np.testing.assert_allclose(score, probe['score'], rtol=1e-7, atol=1e-8)
            np.testing.assert_allclose(probability, probe['probability'], rtol=1e-7, atol=1e-8)
        print('Persistent model/preprocessing reload verified without retraining.')
    else:
        print('Persistent transfer/reload remains incomplete:', persistent)


# 9. Business interpretation and handoff

State whether the selected frozen text-similarity model recovers known clicked articles in a catalogue top-100 shortlist, which information influenced representative recommendations, and what evidence supports keeping it. Do not compare its new catalogue metrics directly with the old E5 recorded-slate AUC/nDCG results.

Bring complete run bundles to the reranking/evaluation owner. Cross-model comparisons require identical requests, shared input versions and eligible catalogues. Stage 2 can apply continuity, freshness and diversity and produce the final five; it cannot recover a relevant article omitted by Stage 1.

Report catalogue versus embedding coverage separately, excluded known clicks, empty pools, recorded-slate diagnostics, runtime/storage and any sampled scope. Training predictions are in-sample, not out-of-fold stacking inputs. Repeated validation is development evidence. Retention and causal engagement gains have not been measured.

**Implementation status:** the shared inputs exist only on Drive and have not been run in this local environment. The helper/export/reload implementation is verified with synthetic fixtures; no synthetic scores are presented as project results. Real shared-data feasibility, catalogue performance and persistent handoff remain unverified until this notebook runs against those files.


In [ ]:
if RESULT is not None:
    explanation = json.loads((WORK / 'validation_explanation.json').read_text())
    display(pd.json_normalize(explanation['top_five']))
    print('Validation request:', explanation['behaviour_id'])
    print(explanation['note'])
    print('Run bundle:', RESULT['persistence'].get('persistent_location', str(WORK)))
    print('Sample scope:', CONFIG['scope'])
    print('Test metrics: unavailable (official test has no click labels).')
